In [1]:
import os
import re
import json
import time
import hashlib
from urllib.parse import urlparse, unquote

import pandas as pd
import requests
from tqdm import tqdm

In [2]:
INPUT_CSV = "E:/2026_capstone/policy_data/policy_data_links.csv"
OUT_DIR = "E:/2026_capstone/policy_data/pdf_data/data"
OUT_META_CSV = "E:/2026_capstone/policy_data/pdf_data/metadata/policy_metadata.csv"
OUT_META_JSONL = "E:/2026_capstone/policy_data/pdf_data/metadata/policy_metadata.jsonl"

REQUEST_TIMEOUT = 30
MAX_RETRIES = 3
SLEEP_BETWEEN = 0.5

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; PolicyDownloader/1.0)"
}

SPLIT_TOKEN = ", "

In [3]:
def ensure_dir(path:str):
    os.makedirs(path, exist_ok=True)

def norm_colnames(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    return df

def split_multi(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    s = str(value).strip()
    if not s or s.lower() == "nan":
        return []
    return [x.strip() for x in s.split(SPLIT_TOKEN) if x.strip()]

def safe_filename(s: str, max_len: int = 160) -> str:
    s = unquote(s)
    s = re.sub(r"[\\/:*?\"<>|]+", "_", s)
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) > max_len:
        s = s[:max_len].rstrip()
    return s or "file"

def sha1(text:str) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()

def request_with_retries(method, url, session: requests.Session, **kwargs):
    last_err = None
    for i in range(MAX_RETRIES):
        try:
            resp = session.request(method, url, timeout=REQUEST_TIMEOUT, allow_redirects=True, **kwargs)
            return resp
        except Exception as e:
            last_err = str(e)
            time.sleep(0.8 * (i + 1))
    raise RuntimeError(last_err or "request failed")

def is_direct_pdf_link(url:str) -> bool:
    try:
        path = urlparse(url).path.lower()
    except Exception:
        return False
    return ".pdf" in path

def get_url_basename(url:str) -> str:
    path = urlparse(url).path
    base = os.path.basename(path.rstrip("/"))
    return base or "document.pdf"

def build_policy_id(row:dict) -> str:
    link = str(row.get("link", "")).strip()
    name = str(row.get("Policy Name", "")).strip()
    return sha1((name + "||" + link).strip())

In [5]:
ensure_dir(OUT_DIR)

def read_csv_fallback(path: str) -> pd.DataFrame:
    encodings_to_try = ["utf-8", "utf-8-sig", "cp1252", "latin1", "gbk", "gb18030"]
    last_err = None
    for enc in encodings_to_try:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as e:
            last_err = e
    # 最后兜底：强行替换坏字符（尽量不建议，但能跑起来）
    return pd.read_csv(path, encoding="latin1")

df = read_csv_fallback(INPUT_CSV)
df = norm_colnames(df)

if "Links" not in df.columns:
    raise ValueError(f"CSV must contain a 'Link' column. Found: {list(df.columns)}")

session = requests.Session()
session.headers.update(HEADERS)

records = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Downloading PDFs (skip non-PDF links)"):
    rowd = row.to_dict()

    link = str(rowd.get("Links", "")).strip()
    policy_id = build_policy_id(rowd)

    states = split_multi(rowd.get("State", ""))
    counties = split_multi(rowd.get("County Name", ""))
    cities = split_multi(rowd.get("City Name", ""))
    tribes = split_multi(rowd.get("Tribe Name", ""))

    policy_level = str(rowd.get("Policy Level", "")).strip()
    policy_type = str(rowd.get("Policy Type (Climate/Energy/Weather/etc.)", "")).strip()

    rec = {
        "row_index": int(idx),
        "policy_id": policy_id,
        "source_url": link,
        "final_url": "",
        "downloaded": 0,
        "needs_manual": 0,
        "http_status": "",
        "content_type": "",
        "file_path": "",
        "error": "",
        "state_list": states,
        "county_list": counties,
        "city_list": cities,
        "tribe_list": tribes,
        "policy_level": policy_level,
        "policy_type": policy_type,
    }

    # 空链接
    if not link or link.lower() == "nan":
        rec["needs_manual"] = 1
        rec["error"] = "empty link"
        records.append(rec)
        continue

    # 不是 PDF 直链：直接跳过并标记人工处理
    if not is_direct_pdf_link(link):
        rec["needs_manual"] = 1
        rec["error"] = "non-pdf direct link (skipped); manual download required"
        records.append(rec)
        continue

    # 保存目录：按 state / level 分桶
    bucket_state = states[0] if states else "UnknownState"
    bucket_level = policy_level if policy_level else "UnknownLevel"
    out_bucket = os.path.join(OUT_DIR, safe_filename(bucket_state), safe_filename(bucket_level))
    ensure_dir(out_bucket)

    try:
        resp = request_with_retries("GET", link, session=session, stream=True)
        rec["http_status"] = resp.status_code
        rec["final_url"] = resp.url
        rec["content_type"] = (resp.headers.get("Content-Type") or "").split(";")[0].strip().lower()

        # 文件名：policy_id前10位 + URL basename
        base = safe_filename(get_url_basename(resp.url))
        if not base.lower().endswith(".pdf"):
            base += ".pdf"
        filename = f"{policy_id[:10]}__{base}"
        out_path = os.path.join(out_bucket, filename)

        # 下载
        with open(out_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 256):
                if chunk:
                    f.write(chunk)

        rec["downloaded"] = 1
        rec["file_path"] = out_path

        time.sleep(SLEEP_BETWEEN)

    except Exception as e:
        rec["needs_manual"] = 1
        rec["error"] = str(e)

    records.append(rec)

# 输出 metadata
meta_df = pd.DataFrame(records)

def join_list(x):
    return "|".join(x) if isinstance(x, list) else ""

meta_df["state_flat"] = meta_df["state_list"].apply(join_list)
meta_df["county_flat"] = meta_df["county_list"].apply(join_list)
meta_df["city_flat"] = meta_df["city_list"].apply(join_list)
meta_df["tribe_flat"] = meta_df["tribe_list"].apply(join_list)

meta_df.to_csv(OUT_META_CSV, index=False, encoding="utf-8-sig")
with open(OUT_META_JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"\nDone.\n- Files saved under: {OUT_DIR}\n- Metadata CSV: {OUT_META_CSV}\n- Metadata JSONL: {OUT_META_JSONL}")
print("Review rows with needs_manual=1 for manual downloading.")




Done.
- Files saved under: E:/2026_capstone/policy_data/pdf_data/data
- Metadata CSV: E:/2026_capstone/policy_data/pdf_data/metadata/policy_metadata.csv
- Metadata JSONL: E:/2026_capstone/policy_data/pdf_data/metadata/policy_metadata.jsonl
Review rows with needs_manual=1 for manual downloading.
